In [1]:
from _src.Utils.CompositionLoader import CompositionExcelLoader
from _src.Composition.CompositionV2 import Composition
from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
from _src.VLE.PhaseEquilibriumNewtonV2 import PhaseEquilibriumNewton
from _src.Utils.FluidPropertiesCalculatorV2 import FluidPropertiesCalculator
from _src.PhaseDiagram.SaturatuinPressureV2 import SaturationPressureCalculation
from _src.PhaseDiagram.PhaseEnvelopeV2 import PhaseDiagram

In [2]:
xl_loader = CompositionExcelLoader(r'/Users/daniilsidorenko/Desktop/PVT_TSU/diss/krsnln.xlsx')
comp_dict = xl_loader.load(header=True, sheet = 'to_model')

In [3]:
composition = Composition(comp_dict)
composition.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'kesler lee',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [4]:
composition.edit_bip_for_components(component1='C1', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='C1', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='C2', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='C2', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='C3', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='C3', component2='C6', bip = 0)

composition.edit_bip_for_components(component1='iC4', component2='C2', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='C3', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='iC4', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='nC4', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='nC5', bip = 0)
composition.edit_bip_for_components(component1='iC4', component2='C6', bip = 0)

In [ ]:
composition.composition_data

In [4]:
condition_p = 1000
condition_t = 100 + 273.15

In [ ]:
phase_stab = TwoPhaseStabilityTest(composition, condition_p, condition_t)
phase_stab.calculate_phase_stability()
phase_stab.stable

In [ ]:
phase_stab.xi_l

In [ ]:
phase_stab.yi_v

In [ ]:
phase_equil = PhaseEquilibriumNewton(composition, condition_p, condition_t, phase_stab.k_values_for_flash)
flash_data = phase_equil.find_solve_loop()

In [ ]:
flash_data

Проблема: FluidProperties от Дениса будто некорректный, надо проверить на старом модуле

In [44]:
fluid_props = FluidPropertiesCalculator(composition=flash_data['xi_l'], composition_properties= composition.composition_data,
                                        p= condition_p, T=condition_t, eos_object = phase_equil.eos_liquid)

In [ ]:
fluid_props.density

In [ ]:
fluid_props.viscosity

In [ ]:
fluid_props.z_shift

### Работа через интерфейс  Model

In [4]:
from _src.CompositionalModel.CompositionalModelV2 import CompositionalModel
from _src.Utils.Conditions import Conditions

In [5]:
model = CompositionalModel(composition)

In [6]:
conds = Conditions(100, 330)

In [ ]:
model.flash(conditions=conds)

In [ ]:
model.composition.composition_data

In [ ]:
model.flashV2(conds)

In [ ]:
sat_p_new = SaturationPressureCalculation(composition_object = composition,
                                          p_max = 600, temp = 120)
sat_p_new.calculate_saturation_pressure()

In [ ]:
from _src.PhaseDiagram.PhaseEnvelopeV2 import SaturationPressure

sat_p_new = SaturationPressure(composition, p_max=120, temp = 165)
sat_p_new.sp_convergence_loop()

In [ ]:
phase_env_new.plot_phase_diagram()

In [ ]:
phase_env_new.get_phase_diagram_data()

# Qwen Pcrit

### QWEN PSAT

In [50]:
import numpy as np
import logging
from _src.EOS.BrusilovskiyEOSV2 import BrusilovskiyEOS
from _src.Composition.CompositionV2 import Composition
from _src.Utils.Errors import ConvergenceError

# =============================================================================
# ЛОГГЕР
# =============================================================================

def _setup_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setLevel(level)
        formatter = logging.Formatter('[%(levelname)-8s] %(name)s: %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.propagate = False
    return logger

logger = _setup_logger('src.BubblePointPressure')


class BubblePointPressure:
    """
    Расчет давления насыщения (bubble point) методом Ньютона-Рафсона.
    Реализация алгоритма Table 6.1 (Brusilovsky).
    
    ⚠️ Все давления — в БАРАХ.
    """
    
    def __init__(self, composition: Composition, T: float):
        self._composition = composition
        self.T = float(T)
        
        # Векторизация состава и свойств
        self.zi = composition.composition
        self._components = tuple(self.zi.keys())
        self._nc = len(self._components)
        
        comp_data = composition.composition_data
        self._z = np.fromiter((self.zi[c] for c in self._components), dtype=np.float64, count=self._nc)
        self._omega = np.fromiter((comp_data['acentric_factor'][c] for c in self._components), dtype=np.float64, count=self._nc)
        self._Pc = np.fromiter((comp_data['critical_pressure'][c] for c in self._components), dtype=np.float64, count=self._nc)
        self._Tc = np.fromiter((comp_data['critical_temperature'][c] for c in self._components), dtype=np.float64, count=self._nc)
        
        # Результаты
        self.P = None   # бар
        self.K = None
        self.y = None
        self.converged = False
        self.iterations = 0

    def _to_dict(self, arr):
        return {c: float(arr[i]) for i, c in enumerate(self._components)}
    
    def _calc_Psat_initial_Dong_Lienhard(self) -> float:
        """
        Начальное приближение давления насыщения по корреляции Dong & Lienhard (1986).
        Использует псевдокритические свойства смеси.
        """
        # Псевдокритические параметры смеси (мольное усреднение)
        Tc_mix = float(np.sum(self._z * self._Tc))  # псевдокритическая температура
        Pc_mix = float(np.sum(self._z * self._Pc))  # псевдокритическое давление
        omega_mix = float(np.sum(self._z * self._omega))  # псевдо-ацентрический фактор
        
        # Приведенная температура
        Tr = self.T / Tc_mix
        
        # Проверка физичности
        if Tr <= 0 or Tr >= 1.5:
            logger.warning(f"Tr={Tr:.3f} вне разумных границ. Возвращаем среднее Pc.")
            return Pc_mix
        
        # Корреляция Dong & Lienhard
        ln_Pr_sat = (
            5.37270 * (1 - 1/Tr) + 
            omega_mix * (7.49408 - 11.18177 * Tr**3 + 3.68769 * Tr**6 + 17.92998 * np.log(Tr))
        )
        
        Pr_sat = np.exp(ln_Pr_sat)
        Psat_initial = Pr_sat * Pc_mix
        
        logger.info(f"Начальное приближение (Dong-Lienhard): P₀ = {Psat_initial:.2f} бар "
                f"(Tr={Tr:.3f}, ω_mix={omega_mix:.3f})")
        
        return float(Psat_initial)
    

    def _wilson_K(self, P_bar):
        """Уравнение 6.13: начальные K-факторы"""
        lnK = np.log(self._Pc / P_bar) + 5.373 * (1 + self._omega) * (1 - self._Tc / self.T)
        return np.exp(np.clip(lnK, -15, 15))

    def _get_ln_phi(self, comp_dict, P_bar):
        """Расчет ln(φ_i) через УРС. Возвращает массив или None при сбое."""
        try:
            new_comp = self._composition.new_composition(comp_dict)
            eos = BrusilovskiyEOS(composition=new_comp, p=P_bar, t=self.T)
            eos.calc_eos()
            return eos.fugacity_coef_by_roots[eos._chosen_root_index].copy()
        except Exception:
            return None

    def _inner_loop(self, P_bar, K_init, max_inner=50, tol_inner=1e-5):
        """
        Внутренний цикл алгоритма (шаги 3-5): стабилизация K при фиксированном P.
        Возвращает (K_final, y_dict, success).
        """
        K = K_init.copy()
        
        for it in range(max_inner):
            # Шаг 3: состав пара
            y = self._z * K
            y = np.maximum(y, 1e-12)
            y /= np.sum(y)
            y_dict = self._to_dict(y)
            
            # Шаг 4: фугитивности
            ln_phi_L = self._get_ln_phi(self.zi, P_bar)
            ln_phi_V = self._get_ln_phi(y_dict, P_bar)
            if ln_phi_L is None or ln_phi_V is None:
                return K, y_dict, False
            
            # Шаг 5: обновление K (уравнение 6.7)
            K_new = np.exp(ln_phi_L - ln_phi_V)
            K_new = np.clip(K_new, 1e-8, 1e8)
            
            # Проверка сходимости внутреннего цикла
            if np.max(np.abs(K_new / K - 1)) < tol_inner:
                return K_new, y_dict, True
            K = K_new
            
        return K, y_dict, True  # Возвращаем последнее значение даже если не сошлось

    def _calc_F(self, K):
        """Шаг 6: F = Σ(z_i * K_i) - 1"""
        return float(np.sum(self._z * K) - 1.0)

    def _calc_dF_dP(self, P_bar, K):
        """Шаг 7: численная производная dF/dP"""
        dP = 0.1  # бар
        K_plus, _, ok1 = self._inner_loop(P_bar + dP, K)
        K_minus, _, ok2 = self._inner_loop(P_bar - dP, K)
        if not (ok1 and ok2):
            return None
        F_plus = self._calc_F(K_plus)
        F_minus = self._calc_F(K_minus)
        return (F_plus - F_minus) / (2 * dP)

    def calculate(self, P_initial_bar=None, tol=1e-6, max_iter=50, damping=0.3):
        # Шаг 1: начальное приближение давления
        if P_initial_bar is None:
            # Используем корреляцию Dong-Lienhard вместо простого среднего Pc
            self.P = self._calc_Psat_initial_Dong_Lienhard()
        else:
            self.P = float(P_initial_bar)
        logger.info(f"Старт: P₀ = {self.P:.2f} бар")

        # Шаг 2: начальные K по Вильсону
        self.K = self._wilson_K(self.P)

        # Основной цикл (шаги 3-9)
        for it in range(max_iter):
            self.iterations = it + 1
            
            # Шаги 3-5: стабилизация K при текущем P
            self.K, y_dict, ok = self._inner_loop(self.P, self.K)
            if not ok:
                logger.warning(f"Ит {it+1}: сбой УРС при P={self.P:.2f} бар")
                self.P *= 0.99  # Мягкий откат
                continue
            
            self.y = y_dict
            
            # Шаг 6: функция сходимости
            F = self._calc_F(self.K)
            logger.debug(f"Ит {it+1}: P={self.P:.2f} бар, F={F:+.6e}")
            
            # Проверка сходимости
            if abs(F) < tol:
                self.converged = True
                logger.info(f"✅ Сходимость за {it+1} итераций | P = {self.P:.3f} бар")
                return self.P
            
            # Шаг 7: производная
            dF_dP = self._calc_dF_dP(self.P, self.K)
            if dF_dP is None or abs(dF_dP) < 1e-12:
                logger.debug(f"Ит {it+1}: dF/dP неопределена, пропускаем шаг")
                continue
            
            # Шаг 8: обновление давления (Ньютон с демпфированием)
            delta = -F / dF_dP
            max_step = damping * self.P
            if abs(delta) > max_step:
                delta = max_step * np.sign(delta)
            self.P = np.clip(self.P + delta, 1.0, 2000.0)  # бар

        # Не сошлось
        self.converged = False
        raise ConvergenceError(f"Не сошлось за {max_iter} итераций | P={self.P:.2f} бар, F={F:.6e}")

    # =============================================================================
    # СВОЙСТВА
    # =============================================================================
    
    @property
    def bubble_point_pressure(self) -> float:
        if not self.converged:
            raise RuntimeError("Расчет не выполнен или не сошелся")
        return self.P  # бар

    @property
    def bubble_point_pressure_MPa(self) -> float:
        return self.P / 10.0

    @property
    def bubble_point_pressure_atm(self) -> float:
        return self.P / 1.01325

    @property
    def k_factors(self) -> dict:
        return self._to_dict(self.K) if self.K is not None else {}

    @property
    def vapour_composition(self) -> dict:
        return self.y if self.y is not None else {}

In [52]:
bubble = BubblePointPressure(composition, T=373.0)
P_bubble = bubble.calculate(tol= 1e-9)
print(f"Bubble point: {P_bubble/1e6:.2f} МПа")
print(f"K-факторы: {bubble.k_factors}")
print(f"Состав пара: {bubble.vapour_composition}")

[INFO    ] src.BubblePointPressure: Начальное приближение (Dong-Lienhard): P₀ = 23.26 бар (Tr=0.934, ω_mix=0.255)
[INFO    ] src.BubblePointPressure: Старт: P₀ = 23.26 бар
[INFO    ] src.BubblePointPressure: ✅ Сходимость за 12 итераций | P = 178.993 бар


Bubble point: 0.00 МПа
K-факторы: {'N2': 2.927058974176667, 'CO2': 1.1157029376492305, 'C1': 1.916748290570751, 'C2': 1.0583095214072036, 'C3': 0.7062263123754039, 'iC4': 0.5419509890864049, 'nC4': 0.4735325775871706, 'iC5': 0.34851117781267443, 'nC5': 0.3292813541682886, 'C6': 0.24362191023508512, 'C7': 0.16900157512484004, 'C8': 0.12454867807834903, 'C9': 0.08906864175163862, 'C10': 0.06517134455836521, 'C11': 0.048581053667045127, 'C12': 0.035271371304890047, 'C13': 0.02586886451747821, 'C14': 0.018534922848288015, 'C15': 0.013041383439102912, 'C16': 0.00944111157013774, 'C17': 0.006785144479260693, 'C18': 0.005148988228952657, 'C19': 0.003996909162226297, 'C20': 0.003072721652032128, 'C21': 0.002241180736909187, 'C22': 0.0016725206402622786, 'C23': 0.0012571190881149844, 'C24': 0.000958269267792894, 'C25': 0.0007298058833363176, 'C26': 0.0005451767114485736, 'C27': 0.0003999770514024502, 'C28': 0.0003028397528169052, 'C29': 0.00022910060641236922, 'C30': 0.00017300059267351633, 'C3

### Реализация через бисекцию

In [45]:
import numpy as np
import logging
from _src.Composition.CompositionV2 import Composition

# =============================================================================
# ЛОГГЕР
# =============================================================================

def _setup_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setLevel(level)
        formatter = logging.Formatter('[%(levelname)-8s] %(name)s: %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.propagate = False
    return logger

logger = _setup_logger('src.BubblePointPressure')


class BubblePointPressureBisect:
    """
    Расчет давления насыщения (bubble point) методом бисекции 
    на основе теста фазовой стабильности.
    
    ⚠️ Все давления — в БАРАХ.
    
    Алгоритм:
    1. Задаем диапазон [P_min, P_max], где гарантированно есть решение
    2. На каждой итерации:
       - Делаем тест стабильности при текущем P
       - Если S_v < 1: P_lo = P (нужно выше)
       - Если S_v > 1: P_hi = P (нужно ниже)
       - Если |S_v - 1| < tol: сошлись!
    """
    
    def __init__(self, composition: Composition, T: float):
        self._composition = composition
        self.T = float(T)
        
        # Состав
        self.zi = composition.composition
        self._components = tuple(self.zi.keys())
        self._nc = len(self._components)
        
        # Результаты
        self.P = None      # бар
        self.S_v = None    # значение стабильности в точке решения
        self.stability = None  # объект теста стабильности
        self.converged = False
        self.iterations = 0

    def _run_stability_test(self, P_bar: float):
        """Запускает тест стабильности при заданном давлении."""
        from _src.PhaseStability.TwoPhaseStabilityTestV3 import TwoPhaseStabilityTest
        stability = TwoPhaseStabilityTest(self._composition, P_bar, self.T)
        stability.calculate_phase_stability()
        return stability

    def _find_bracket(self, P_min: float, P_max: float, max_attempts: int = 20) -> tuple:
        """
        Находит диапазон [P_lo, P_hi], где S_v меняет знак относительно 1.0.
        Возвращает (P_lo, P_hi, stab_lo, stab_hi) или raises ConvergenceError.
        """
        from _src.Utils.Errors import ConvergenceError
        
        # Проверяем нижнюю границу: должна быть стабильная жидкость (S_v < 1)
        stab_lo = self._run_stability_test(P_min)
        
        # Если даже при минимальном давлении S_v > 1 — сужаем
        attempts = 0
        while stab_lo.S_v is not None and stab_lo.S_v > 1.0 and attempts < max_attempts:
            logger.debug(f"P_min={P_min:.3f} бар: S_v={stab_lo.S_v:.3f} > 1, сужаем")
            P_min *= 0.5
            if P_min < 1e-4:
                break
            stab_lo = self._run_stability_test(P_min)
            attempts += 1
        
        # Проверяем верхнюю границу: должна быть нестабильность (S_v > 1)
        stab_hi = self._run_stability_test(P_max)
        attempts = 0
        while stab_hi.S_v is not None and stab_hi.S_v < 1.0 and attempts < max_attempts:
            logger.debug(f"P_max={P_max:.3f} бар: S_v={stab_hi.S_v:.3f} < 1, расширяем")
            P_max *= 1.5
            if P_max > 5000.0:
                break
            stab_hi = self._run_stability_test(P_max)
            attempts += 1
        
        # Валидация бракетинга
        if stab_lo.S_v is None or stab_hi.S_v is None:
            raise ConvergenceError("Не удалось рассчитать S_v на границах диапазона")
        
        if stab_lo.S_v >= 1.0 and stab_hi.S_v <= 1.0:
            raise ConvergenceError(
                f"Не удалось найти бракетинг: "
                f"S_v(P_min={P_min:.2f})={stab_lo.S_v:.3f}, "
                f"S_v(P_max={P_max:.2f})={stab_hi.S_v:.3f}. "
                "Возможно, температура выше критической."
            )
        
        logger.info(f"✅ Бракетинг: [{P_min:.2f}, {P_max:.2f}] бар, "
                   f"S_v=[{stab_lo.S_v:.3f}, {stab_hi.S_v:.3f}]")
        
        return P_min, P_max, stab_lo, stab_hi

    def calculate(self, P_min_bar: float = 0.01, P_max_bar: float = None, 
                  tol: float = 1e-4, max_iter: int = 100) -> float:
        """
        Расчет давления насыщения методом бисекции по критерию стабильности.
        
        Parameters
        ----------
        P_min_bar : float
            Нижняя граница поиска, бар (по умолчанию 0.01).
        P_max_bar : float, optional
            Верхняя граница поиска, бар. Если None, оценивается автоматически.
        tol : float
            Критерий сходимости по |S_v - 1|.
        max_iter : int
            Максимальное число итераций бисекции.
            
        Returns
        -------
        float
            Давление насыщения, бар.
        """
        from _src.Utils.Errors import ConvergenceError
        
        # Автооценка верхней границы, если не задана
        if P_max_bar is None:
            comp_data = self._composition.composition_data
            Pc_avg = np.mean([comp_data['critical_pressure'][c] for c in self._components])
            P_max_bar = np.clip(Pc_avg * 5.0, 100.0, 2000.0)
            logger.info(f"Верхняя граница оценена: {P_max_bar:.2f} бар")
        
        logger.info(f"Поиск bubble point: [{P_min_bar:.2f}, {P_max_bar:.2f}] бар, T={self.T:.2f} K")
        
        # Шаг 1: находим бракетинг
        P_lo, P_hi, stab_lo, stab_hi = self._find_bracket(P_min_bar, P_max_bar)
        
        # Шаг 2: бисекция
        for it in range(max_iter):
            self.iterations = it + 1
            P_mid = 0.5 * (P_lo + P_hi)
            
            # Тест стабильности
            stability = self._run_stability_test(P_mid)
            
            # Если смесь стабильна — точно ниже bubble point
            if stability.stable:
                logger.debug(f"Ит {it+1}: P={P_mid:.2f} бар — стабильная (однофазная)")
                P_lo = P_mid
                continue
            
            S_v = stability.S_v
            logger.debug(f"Ит {it+1}: P={P_mid:.2f} бар, S_v={S_v:.6f}")
            
            # Проверка сходимости
            if abs(S_v - 1.0) < tol:
                self.converged = True
                self.P = P_mid
                self.S_v = S_v
                self.stability = stability
                logger.info(f"✅ Сходимость за {it+1} итераций | "
                           f"P_bubble = {P_mid:.3f} бар, S_v={S_v:.6f}")
                return P_mid
            
            # Обновление границ
            if S_v < 1.0:
                P_lo = P_mid  # нужно выше
            else:
                P_hi = P_mid  # нужно ниже
            
            # Проверка на схлопывание диапазона
            if (P_hi - P_lo) < 1e-8 * P_mid:
                logger.debug(f"Диапазон сузился до {(P_hi - P_lo):.2e} бар")
                break
        
        # Не сошлось
        self.converged = False
        final_Sv = stability.S_v if stability else None
        raise ConvergenceError(
            f"Не сошлось за {max_iter} итераций | "
            f"Диапазон=[{P_lo:.2f}, {P_hi:.2f}] бар, "
            f"последняя S_v={final_Sv:.6f}, P={P_mid:.2f} бар"
        )

    # =============================================================================
    # СВОЙСТВА
    # =============================================================================
    
    @property
    def bubble_point_pressure(self) -> float:
        """Давление насыщения, бар"""
        if not self.converged:
            raise RuntimeError("Расчет не выполнен или не сошелся. Вызовите calculate()")
        return self.P
    
    @property
    def bubble_point_pressure_MPa(self) -> float:
        """Давление насыщения, МПа"""
        return self.P / 10.0
    
    @property
    def bubble_point_pressure_atm(self) -> float:
        """Давление насыщения, атм"""
        return self.P / 1.01325
    
    @property
    def stability_result(self):
        """Объект теста стабильности в точке решения"""
        return self.stability
    
    @property
    def k_factors(self) -> dict:
        """K-факторы в точке насыщения (из теста стабильности)"""
        if self.stability and self.stability.k_vals_for_sat_point_calculation:
            return self.stability.k_vals_for_sat_point_calculation
        return {}
    
    @property
    def vapour_composition(self) -> dict:
        """Состав пара в точке насыщения"""
        if self.stability and self.stability.yi_v:
            return self.stability.yi_v
        return {}

In [49]:
bubble = BubblePointPressureBisect(composition, T=373.0)
P_bubble = bubble.calculate(P_max_bar=400, tol= 1e-5)
print(f"Bubble point: {P_bubble/1e6:.2f} МПа")
print(f"K-факторы: {bubble.k_factors}")
print(f"Состав пара: {bubble.vapour_composition}")

[INFO    ] src.BubblePointPressure: Поиск bubble point: [0.01, 400.00] бар, T=373.00 K
[INFO    ] src.BubblePointPressure: ✅ Бракетинг: [0.00, 400.00] бар, S_v=[1810894.993, 1.000]


ConvergenceError: Не сошлось за 100 итераций | Диапазон=[400.00, 400.00] бар, последняя S_v=1.000005, P=400.00 бар

### QWEN CRITP

In [ ]:
phase_env = PhaseEnvelopeTracer(composition)
phase_env.trace(T0=280, P0=5)

In [ ]:
import pandas as pd
history = phase_env.trace(T0=280.0, P0=5.0, h=2.0, max_steps=200)

df = pd.DataFrame(history)
print(f"Успешно пройдено точек: {len(df)}")
print(f"Последняя точка: T={df.iloc[-1]['T']:.2f} K, P={df.iloc[-1]['P']:.4f} bar")

In [ ]:
phase_env.branch

In [ ]:
tracer = PhaseEnvelopeTracer(composition, branch='bubble', debug=True)
history = tracer.trace(
    T0=280.0, 
    P0=5.0, 
    h=1.0,           # Меньший начальный шаг
    h_min=0.05,      # Более мелкий минимальный шаг
    max_steps=300,
    direction='increasing_T'  # Явно: растём по температуре
)

# Анализ результатов
df = pd.DataFrame(history)
print(f"\n✅ Успешно: {len(df)} точек")
print(f"Диапазон: T=[{df['T'].min():.1f}, {df['T'].max():.1f}] K, "
      f"P=[{df['P'].min():.2f}, {df['P'].max():.2f}] bar")

# График
if len(df) > 2:
    plt.figure(figsize=(9,6))
    plt.plot(df['T'], df['P'], 'o-', markersize=3, linewidth=1.5)
    plt.yscale('log')
    plt.xlabel('Temperature, K', fontsize=11)
    plt.ylabel('Pressure, bar (log)', fontsize=11)
    plt.grid(True, which='both', alpha=0.3)
    plt.title('Phase Envelope (Bubble Point)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [107]:
import numpy as np
from _src.Composition.CompositionV2 import Composition
from _src.EOS.BrusilovskiyEOSV2 import BrusilovskiyEOS

class CriticalPointCalculator:
    """
    Стабилизированный расчет критической точки по Michelsen (1994).
    Устранены: осцилляции, зависимость от P_start, плохая обусловленность.
    """
    def __init__(self, composition: Composition, P_start: float = None):
        self.base_comp = composition
        self.components = tuple(composition.composition.keys())
        self.nc = len(self.components)
        self.z_arr = np.array([composition.composition[c] for c in self.components])
        
        # Данные компонент
        self._comp_data = composition.composition_data
        self._Tc_arr = np.array([self._comp_data['critical_temperature'][c] for c in self.components])
        self._Pc_arr = np.array([self._comp_data['critical_pressure'][c] for c in self.components])
        self._omega_arr = np.array([self._comp_data['acentric_factor'][c] for c in self.components])
        
        # Автоматический выбор P_start вблизи ожидаемой критической области
        if P_start is None:
            P_start = 0.6 * np.sum(self.z_arr * self._Pc_arr)  # ~30-50 атм для газов
        self.P_start = P_start
        self.trace_history = []

    def _create_eos(self, comp_dict: dict, T: float, P: float) -> BrusilovskiyEOS:
        eos = BrusilovskiyEOS(composition=self.base_comp.new_composition(comp_dict), p=P, t=T)
        eos.calc_eos()
        return eos

    def _get_ln_phi(self, eos: BrusilovskiyEOS, x_arr: np.ndarray, P: float) -> np.ndarray:
        return eos.fugacities - np.log(np.clip(x_arr, 1e-15, 1.0) * P)

    def _solve_bubble_point(self, P: float) -> tuple:
        T = np.clip(0.9 * np.sum(self.z_arr * self._Tc_arr), 150.0, 0.95 * np.max(self._Tc_arr))
        K = (self._Pc_arr / P) * np.exp(5.37 * (1.0 + self._omega_arr) * (1.0 - self._Tc_arr / T))
        K = np.clip(K, 1e-6, 1e6)

        for _ in range(50):
            y_arr = (self.z_arr * K) / np.sum(self.z_arr * K)
            eos_z = self._create_eos(dict(zip(self.components, self.z_arr)), T, P)
            eos_y = self._create_eos(dict(zip(self.components, y_arr)), T, P)
            
            K_new = np.exp(self._get_ln_phi(eos_z, self.z_arr, P) - self._get_ln_phi(eos_y, y_arr, P))
            K_new = np.clip(K_new, 1e-6, 1e6)
            if np.max(np.abs(K_new - K)) < 1e-4:
                K = K_new
                break
            K = K_new

            F = np.sum(self.z_arr * K) - 1.0
            if abs(F) < 1e-6: break

            dT = max(0.2, 1e-3 * T)
            K_p = np.exp(self._get_ln_phi(self._create_eos(dict(zip(self.components, self.z_arr)), T+dT, P), self.z_arr, P) - 
                         self._get_ln_phi(self._create_eos(dict(zip(self.components, y_arr)), T+dT, P), y_arr, P))
            dF_dT = (np.sum(self.z_arr * K_p) - 1.0 - F) / dT
            if abs(dF_dT) < 1e-8: T += np.sign(F) * 2.0
            else: T -= F / dF_dT
            T = np.clip(T, 150.0, 1.1 * np.max(self._Tc_arr))

        return T, y_arr, np.log(np.clip(K, 1e-12, 1e12))

    def _solve_theta(self, alpha: float, ln_K_ref: np.ndarray, beta: float = 0.5) -> float:
        theta = 0.0
        for _ in range(30):
            ln_K = alpha * ln_K_ref + theta
            K = np.exp(np.clip(ln_K, -50, 50))
            denom = 1.0 + beta * (K - 1.0)
            denom = np.where(np.abs(denom) < 1e-12, 1e-12 * np.sign(denom), denom)
            f = np.sum(self.z_arr * K / denom) - 1.0
            if abs(f) < 1e-10: return theta
            df_dtheta = np.sum(self.z_arr * K / denom - beta * self.z_arr * K**2 / denom**2)
            if abs(df_dtheta) < 1e-12: break
            theta -= f / df_dtheta
        return theta

    def _calc_f_and_jacobian(self, T: float, P: float, y_arr: np.ndarray) -> tuple:
        dict_y = dict(zip(self.components, y_arr))
        dict_z = dict(zip(self.components, self.z_arr))
        
        eos_y = self._create_eos(dict_y, T, P)
        eos_z = self._create_eos(dict_z, T, P)
        ln_phi_y = self._get_ln_phi(eos_y, y_arr, P)
        ln_phi_z = self._get_ln_phi(eos_z, self.z_arr, P)
        
        d = np.log(np.clip(y_arr, 1e-15, 1.0)) + ln_phi_y - np.log(np.clip(self.z_arr, 1e-15, 1.0)) - ln_phi_z
        f1 = np.dot(y_arr, d)
        f2 = np.dot(self.z_arr, d)
        
        # Логарифмическое масштабирование шагов: h = ε * T, ε * P
        hT, hP = 1e-4 * T, 1e-4 * P
        
        def get_dlnphi_dlnT(eos_obj, x_arr):
            lp_p = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T*(1+hT), P), x_arr, P)
            lp_m = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T*(1-hT), P), x_arr, P)
            return (lp_p - lp_m) / (2*hT)
            
        def get_dlnphi_dlnP(eos_obj, x_arr):
            lp_p = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T, P*(1+hP)), x_arr, P*(1+hP))
            lp_m = self._get_ln_phi(self._create_eos(dict(zip(self.components, x_arr)), T, P*(1-hP)), x_arr, P*(1-hP))
            return (lp_p - lp_m) / (2*hP)
            
        dlnphi_dlnT = get_dlnphi_dlnT(eos_y, y_arr) - get_dlnphi_dlnT(eos_z, self.z_arr)
        dlnphi_dlnP = get_dlnphi_dlnP(eos_y, y_arr) - get_dlnphi_dlnP(eos_z, self.z_arr)
        
        # Якобиан для переменных τ=ln T, π=ln P
        J11 = np.dot(y_arr, dlnphi_dlnT)
        J21 = np.dot(self.z_arr, dlnphi_dlnT)
        J12 = np.dot(y_arr, dlnphi_dlnP)
        J22 = np.dot(self.z_arr, dlnphi_dlnP)
        
        return f1, f2, J11, J12, J21, J22

    def calculate(self, alpha_steps=None, max_corrector_iters=15, tol=1e-7) -> dict:
        if alpha_steps is None:
            # Плотная фиксированная сетка. Адаптивная вставка удалена для стабильности итератора.
            alpha_steps = [1.0, 0.7, 0.5, 0.35, 0.25, 0.18, 0.12, 0.08, 0.05, 0.03]
            
        print("[CriticalPoint] Шаг 0: Поиск опорной точки (bubble-point)...")
        T, y_init, ln_K_ref = self._solve_bubble_point(self.P_start)
        P = self.P_start
        print(f"  → Начальная точка: T={T:.2f} K, P={P:.2f} atm")
        
        self.trace_history = [(1.0, T, P)]
        dT_dalpha, dP_dalpha = 0.0, 0.0  # Касательная для предиктора
        
        for i, alpha in enumerate(alpha_steps[1:], 1):
            theta = self._solve_theta(alpha, ln_K_ref, beta=0.5)
            ln_K = alpha * ln_K_ref + theta
            K = np.exp(np.clip(ln_K, -50, 50))
            y_arr = (self.z_arr * K) / np.sum(self.z_arr * K)
            
            # ✅ БЕЗОПАСНЫЙ Секантный предиктор
            if len(self.trace_history) >= 2:
                T_guess = T + dT_dalpha * (self.trace_history[-2][0] - alpha)
                P_guess = P + dP_dalpha * (self.trace_history[-2][0] - alpha)
            else:
                # На первом шаге используем текущую точку как приближение
                T_guess, P_guess = T, P
                
            T_guess = np.clip(T_guess, 150.0, 500.0)
            P_guess = np.clip(P_guess, 1.0, 300.0)

            # Корректор: Damped Newton в логарифмических переменных
            tau, pi = np.log(T_guess), np.log(P_guess)
            f_norm_prev = 1e9
            step_accept = 1.0
            
            for it in range(max_corrector_iters):
                f1, f2, J11, J12, J21, J22 = self._calc_f_and_jacobian(np.exp(tau), np.exp(pi), y_arr)
                f_norm = np.sqrt(f1**2 + f2**2)
                
                if it > 1 and f_norm < tol:
                    break
                    
                det = J11 * J22 - J12 * J21
                if abs(det) < 1e-14: det = 1e-14 * np.sign(det)
                
                dtau = -(J22 * f1 - J12 * f2) / det
                dpi = -(J11 * f2 - J21 * f1) / det
                
                # Trust-region: ограничиваем относительный шаг
                max_dtau, max_dpi = 0.05, 0.10  # ~5% по T, ~10% по P
                if abs(dtau) > max_dtau: dtau = max_dtau * np.sign(dtau)
                if abs(dpi) > max_dpi: dpi = max_dpi * np.sign(dpi)
                
                # Backtracking line search
                step = 1.0
                for _ in range(6):
                    tau_t = tau + step * dtau
                    pi_t = pi + step * dpi
                    T_t, P_t = np.exp(tau_t), np.exp(pi_t)
                    if T_t < 150.0 or P_t < 1.0:
                        step *= 0.5; continue
                    f1_t, f2_t, _, _, _, _ = self._calc_f_and_jacobian(T_t, P_t, y_arr)
                    if np.sqrt(f1_t**2 + f2_t**2) < f_norm:
                        break
                    step *= 0.5
                    
                tau += step * dtau
                pi += step * dpi
                
                if f_norm < f_norm_prev:
                    f_norm_prev = f_norm
                    step_accept = step
                else:
                    step = max(step * 0.2, 1e-4)
                    tau -= step * dtau * 0.5
                    pi -= step * dpi * 0.5

            T, P = np.exp(tau), np.exp(pi)
            
            # Обновляем касательную для следующего предиктора
            prev_alpha = self.trace_history[-1][0]
            dT_dalpha = (T - self.trace_history[-1][1]) / (alpha - prev_alpha)
            dP_dalpha = (P - self.trace_history[-1][2]) / (alpha - prev_alpha)
            
            self.trace_history.append((alpha, T, P))
            print(f"  → α={alpha:.3f}: T={T:.2f} K, P={P:.2f} atm |f|={f_norm:.2e} step_rel={step_accept:.2f}")

        # Квадратичная экстраполяция по α²
        points = self.trace_history[-3:]
        alpha_sq = np.array([p[0]**2 for p in points])
        T_vals = np.array([p[1] for p in points])
        P_vals = np.array([p[2] for p in points])
        
        A = np.vstack([np.ones(3), alpha_sq, alpha_sq**2]).T
        try:
            coeff_T = np.linalg.solve(A, T_vals)
            coeff_P = np.linalg.solve(A, P_vals)
            T_crit, P_crit = coeff_T[0], coeff_P[0]
        except np.linalg.LinAlgError:
            T_crit, P_crit = T_vals[-1], P_vals[-1]
            
        print(f"\n[CriticalPoint] КРИТИЧЕСКАЯ ТОЧКА:")
        print(f"  Tc = {T_crit:.3f} K, Pc = {P_crit:.3f} atm")
        return {'Tc': T_crit, 'Pc': P_crit, 'history': self.trace_history}

In [ ]:

# 2. Запуск расчета
calc = CriticalPointCalculator(composition, P_start=10.0)
result = calc.calculate()

print(f"Критическая точка: Tc = {result['Tc']:.2f} K, Pc = {result['Pc']:.2f} atm")

In [108]:
z = {'C1': 0.943, 'C2': 0.027, 'C3': 0.0074, 'nC4': 0.0049, 
     'nC5': 0.0027, 'C6': 0.0010, 'C7': 0.014}
compt = Composition(z, T_res=100)
compt.evaluate_composition_data({'critical_temperature': 'kesler lee',
                                                    'critical_pressure': 'pedersen',
                                                    'acentric_factor': 'riazi al sahhaf',
                                                    'critical_volume': 'hall yarborough',
                                                    'Kw': 'k watson',
                                                    'shift_parameter': 'jhaveri youngren'})

In [ ]:
calc = CriticalPointCalculator(compt, P_start=10)
result = calc.calculate()

print(f"Критическая точка: Tc = {result['Tc']:.2f} K, Pc = {result['Pc']:.2f} atm")